# Week 7 · Object recognition: free bike spots

**Question:** how many bikes are in a bike rack right now?
**User:** a student who wants to know whether there is still a free spot before walking to the rack.

Object recognition finds every object in a photo and returns a **box**, a **label** and a **confidence** for each one.
My MVP counts the boxes labelled `bicycle` above a confidence threshold: a photo in, a count out.

I took the photos myself with my phone. There are no people in them.

## Setup

In [ ]:
!pip install -q pillow-heif

import os, glob, zipfile, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageOps
import torch, torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights

try:
    import pillow_heif
    pillow_heif.register_heif_opener()  # my iPhone saves photos as .HEIC
except ImportError:
    print("pillow-heif missing, HEIC photos will not open")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| running on:", device)

## Photos

I upload my photos (or one zip with all of them) into Colab.

In [ ]:
PHOTO_DIR = "/content/photos"
os.makedirs(PHOTO_DIR, exist_ok=True)

try:
    from google.colab import files
    for name in files.upload():
        path = os.path.join(PHOTO_DIR, name)
        shutil.move(name, path)
        if name.lower().endswith(".zip"):
            with zipfile.ZipFile(path) as z:
                z.extractall(PHOTO_DIR)
            os.remove(path)
except ImportError:
    PHOTO_DIR = "photos"  # running locally: photos are in a folder next to the notebook
print("files in", PHOTO_DIR, ":", len(os.listdir(PHOTO_DIR)) if os.path.isdir(PHOTO_DIR) else 0)

## Data processing

What happens to a phone photo before the model sees it:
1. **Open:** HEIC is converted and every photo becomes **RGB** (3 colour channels).
2. **Rotate:** phones store the rotation in the EXIF data, not in the pixels, so I apply it first.
3. **Resize:** the longest side becomes at most 1024 px. A 12-megapixel photo is slow and does not help the model.
4. **Tensor:** pixels go from 0–255 to 0–1 and become a tensor of shape `3 × height × width`.
   The model then rescales the photo so the **short side is 800 px** (long side at most 1333 px) and **normalises** the colours
   with the mean and spread of ImageNet photos, which its backbone was first trained on.

Before I had my own photos I tested the pipeline on three example photos from scikit-image.

In [ ]:
MAX_SIDE = 1024
EXTENSIONS = (".jpg", ".jpeg", ".png", ".heic", ".heif", ".webp", ".bmp")

def load_folder(folder, max_side=MAX_SIDE):
    # all photos in the folder (also sub-folders) -> {file name: PIL image}
    images = {}
    paths = sorted(p for p in glob.glob(os.path.join(folder, "**", "*"), recursive=True)
                   if p.lower().endswith(EXTENSIONS) and "__MACOSX" not in p)
    for p in paths:
        try:
            img = ImageOps.exif_transpose(Image.open(p)).convert("RGB")
            img.thumbnail((max_side, max_side))
            name = os.path.basename(p)
            if name in images:
                print("two photos called", name, "- only the last one is kept")
            images[name] = img
        except Exception as e:
            print("could not open", p, "->", e)
    return images

def example_photos():
    from skimage import data
    return {"example_cat.png": Image.fromarray(data.chelsea()),
            "example_coffee.png": Image.fromarray(data.coffee()),
            "example_rocket.png": Image.fromarray(data.rocket())}

photos = load_folder(PHOTO_DIR) if os.path.isdir(PHOTO_DIR) else {}
if not photos:
    print("no photos found, using the example photos")
    photos = example_photos()

print(len(photos), "photos")
for name, img in photos.items():
    print(f"  {name:30s} {img.size[0]}x{img.size[1]}")

In [ ]:
def show_grid(images, cols=4, size=4):
    names = list(images)
    rows = (len(names) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * size, rows * size), squeeze=False)
    for ax in axes.flat: ax.axis("off")
    for ax, n in zip(axes.flat, names):
        ax.imshow(images[n]); ax.set_title(n, fontsize=9)
    plt.tight_layout(); plt.show()

show_grid(photos)

## The model

I use **Faster R-CNN** with a ResNet-50 backbone, pre-trained on **COCO** (330,000 photos, 80 kinds of everyday objects, including `bicycle`).
I do not train it myself: a bike rack is exactly the kind of scene COCO contains, so a pre-trained model is enough for an MVP.

* **Input:** one RGB photo.
* **Output:** a list of detections, each with a box `(x1, y1, x2, y2)` in pixels (y counts from the top), a label and a score between 0 and 1.
* **Two stages:** first it proposes regions that might contain an object, then it classifies each region and refines the box.

I keep every detection with a score of at least 0.05 and apply my own threshold later, so I can test different thresholds without running the model again.

In [ ]:
weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn_v2(weights=weights, box_score_thresh=0.05).to(device).eval()
preprocess = weights.transforms()  # PIL image -> tensor 0-1
CLASSES = weights.meta["categories"]
print(", ".join(c for c in CLASSES if c not in ("__background__", "N/A")))

In [ ]:
@torch.no_grad()
def detect(img):
    # one photo -> one row per detected object, highest score first
    out = model([preprocess(img).to(device)])[0]
    df = pd.DataFrame(out["boxes"].cpu().numpy(), columns=["x1", "y1", "x2", "y2"])
    df["label"] = [CLASSES[i] for i in out["labels"].cpu().numpy()]
    df["score"] = out["scores"].cpu().numpy()
    return df.sort_values("score", ascending=False).reset_index(drop=True)

raw = {name: detect(img) for name, img in photos.items()}
first = next(iter(raw))
print("raw output for", first)
raw[first].head(10)

## The MVP

Count the `bicycle` boxes above the threshold. I start at 0.6 and tune it in the evaluation.
The boxes are numbered (1 = highest score), so I can mark the wrong ones in the ground truth.

In [ ]:
TARGET = ["bicycle"]
THRESHOLD = 0.6
QUESTION = "How many bikes are in the rack?"

assert all(t in CLASSES for t in TARGET)

def mvp(name, threshold=None):
    t = THRESHOLD if threshold is None else threshold
    df = raw[name]
    hits = df[df["label"].isin(TARGET) & (df["score"] >= t)].reset_index(drop=True)
    hits.index = hits.index + 1  # box numbers start at 1
    return hits

def answer(name):
    return f"{QUESTION}  ->  {len(mvp(name))}"

def show_mvp(name, ax=None, threshold=None):
    hits = mvp(name, threshold)
    own = ax is None
    if own: fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(photos[name]); ax.axis("off")
    for i, r in hits.iterrows():
        ax.add_patch(patches.Rectangle((r.x1, r.y1), r.x2 - r.x1, r.y2 - r.y1,
                                       fill=False, edgecolor="lime", linewidth=2))
        ax.text(r.x1, r.y1, f"{i} {r.score:.2f}", color="black", fontsize=8,
                bbox=dict(facecolor="lime", alpha=0.8, pad=1, edgecolor="none"))
    ax.set_title(f"{name}\n{len(hits)} found", fontsize=9)
    if own: plt.show()

def show_all(names=None, cols=3, size=5, threshold=None):
    names = list(photos) if names is None else list(names)
    rows = (len(names) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * size, rows * size), squeeze=False)
    for ax in axes.flat: ax.axis("off")
    for ax, n in zip(axes.flat, names):
        show_mvp(n, ax=ax, threshold=threshold)
    plt.tight_layout(); plt.show()

show_all()
for n in photos:
    print(f"{n:30s} {answer(n)}")

## Ground truth

I counted the bikes on every original photo myself **before** looking at the boxes (`true_count`).
Then I went through the numbered boxes above and wrote down which ones are wrong (`wrong_boxes`): no bike there,
something else, or the same bike boxed twice.

In [ ]:
# helper: prints an empty entry for every photo, which I then filled in below
print("GROUND_TRUTH = {")
for n in photos:
    print(f'    "{n}": {{"true_count": 0, "wrong_boxes": []}},')
print("}")

In [ ]:
GROUND_TRUTH = {
    "example_cat.png":    {"true_count": 0, "wrong_boxes": []},
    "example_coffee.png": {"true_count": 0, "wrong_boxes": []},
    "example_rocket.png": {"true_count": 0, "wrong_boxes": []},
}

missing = [n for n in photos if n not in GROUND_TRUTH]
extra = [n for n in GROUND_TRUTH if n not in photos]
if missing: print("no ground truth for:", missing)
if extra:   print("ground truth for photos that are not loaded:", extra)

## Evaluation

* **Count error (MAE)** = |model count − true count| per photo, averaged. **This is my main metric:** my user only sees the number,
  so what matters is how far that number is off.
* **Precision** = right boxes / all boxes: when the model says "bike", how often is it right?
* **Recall** = right boxes / all real bikes: how many bikes does it find?

Per photo: right boxes = boxes − wrong boxes, missed bikes = true count − right boxes.

In [ ]:
def evaluate(threshold=None, gt=None):
    gt = GROUND_TRUTH if gt is None else gt
    rows = []
    for n in photos:
        if n not in gt: continue
        pred = len(mvp(n, threshold))
        true = gt[n]["true_count"]
        wrong = len([b for b in gt[n].get("wrong_boxes", []) if 1 <= b <= pred])
        tp = pred - wrong
        if tp > true:
            print(f"{n}: {tp} right boxes but true_count {true} - check this photo again")
        rows.append(dict(photo=n, true_count=true, model_count=pred, right=tp, wrong=wrong,
                         missed=max(true - tp, 0), count_error=abs(pred - true)))
    df = pd.DataFrame(rows)
    TP, FP, FN = df.right.sum(), df.wrong.sum(), df.missed.sum()
    summary = dict(precision=TP / (TP + FP) if TP + FP else float("nan"),
                   recall=TP / (TP + FN) if TP + FN else float("nan"),
                   count_MAE=df.count_error.mean(),
                   exact_count=(df.count_error == 0).mean())
    return df, summary

results, summary = evaluate()
display(results)
print(f"threshold {THRESHOLD}: count error {summary['count_MAE']:.2f} per photo · exact {summary['exact_count']:.0%} · "
      f"precision {summary['precision']:.2f} · recall {summary['recall']:.2f}")

### Which threshold?

My `wrong_boxes` only hold for the threshold I judged them at, so this comparison only uses the counts.

In [ ]:
sweep = []
for t in np.arange(0.1, 1.0, 0.05):
    errs = [abs(len(mvp(n, t)) - GROUND_TRUTH[n]["true_count"]) for n in photos if n in GROUND_TRUTH]
    sweep.append(dict(threshold=round(t, 2), count_MAE=np.mean(errs), exact=np.mean([e == 0 for e in errs])))
sweep = pd.DataFrame(sweep)
# lowest count error; on a tie, the most exact photos, then the threshold closest to the current one
sweep["distance"] = (sweep.threshold - THRESHOLD).abs()
best = sweep.sort_values(["count_MAE", "exact", "distance"], ascending=[True, False, True]).iloc[0]

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(sweep.threshold, sweep.count_MAE, marker="o")
ax.axvline(THRESHOLD, color="grey", ls="--", label=f"threshold used: {THRESHOLD}")
ax.set(xlabel="threshold (minimum confidence)", ylabel="count error (MAE)", title="Count error per threshold")
ax.legend(); plt.show()
print(f"lowest count error at threshold {best.threshold} ({best.count_MAE:.2f})")

## Where it fails

The photos with the largest count error. For each one: is it the model's mistake, or is my ground truth wrong?

In [ ]:
worst = results.sort_values("count_error", ascending=False).head(6)
display(worst)
show_all(worst.photo)

In [ ]:
best_photo = results.sort_values(["count_error", "true_count"], ascending=[True, False]).iloc[0].photo
worst_photo = worst.iloc[0].photo
if worst.iloc[0].count_error == 0:
    print("every count is exact on these photos")

print(f"{len(photos)} photos · {', '.join(TARGET)} · threshold {THRESHOLD}")
print(f"count error {summary['count_MAE']:.2f} per photo · exact {summary['exact_count']:.0%} · "
      f"precision {summary['precision']:.2f} · recall {summary['recall']:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
show_mvp(best_photo, ax=axes[0]); show_mvp(worst_photo, ax=axes[1])
axes[0].set_title("Works: " + axes[0].get_title()); axes[1].set_title("Fails: " + axes[1].get_title())
plt.tight_layout(); plt.show()

## Conclusion

*(after running on my photos: what the count error means for my user, why the worst photos fail, and who the app could fail)*